In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_catalogo = spark.table(f"{catalogo}.{esquema_sink}.catalogo_transformed")
df_sms_antiguo = spark.table(f"{catalogo}.{esquema_source}.sms_antiguo")
df_sms_nuevo = spark.table(f"{catalogo}.{esquema_source}.sms_nuevo")

In [0]:
df_catalogo = df_catalogo.drop("ingestion_date")
df_sms_antiguo = df_sms_antiguo.withColumnRenamed(
    "producto",
    "llave"
)

df_sms_antiguo = df_sms_antiguo.dropna(how="all")\
                        .filter((col("control") == "0.0.0.0") | (col("control").isNotNull()))
df_sms_nuevo = df_sms_nuevo.dropna(how="all")


In [0]:
df_sms_antiguo = df_sms_antiguo.withColumn("anio", year(col("fecha_envio")))

df_sms_antiguo = df_sms_antiguo.withColumn("mes", month(col("fecha_envio")))

df_sms_antiguo = df_sms_antiguo.withColumn(
    "periodo",
    date_format(col("fecha_envio"), "yyyy-MM")
)

df_sms_antiguo = df_sms_antiguo.withColumn("plataforma",lit("sms"))

df_sms_antiguo = df_sms_antiguo.withColumn("origen",lit("sms_antiguo"))

df_sms_antiguo = df_sms_antiguo.withColumnRenamed("usuarioSAC","usuario_creacion")

In [0]:
df_sms_antiguo_catalogo = df_catalogo.join(df_sms_antiguo, on=["llave"], how="inner")

In [0]:
df_sms_antiguo_final = df_sms_antiguo_catalogo.select(
    col("anio"),
    col("mes"),
    col("cliente"),
    col("producto"),
    col("fecha_envio"),
    col("usuario_creacion"),
    col("numero"),
    col("control"),
    col("plataforma"),
    col("origen"),
    col("periodo"),
    col("ingestion_date")
)


In [0]:
df_sms_nuevo = df_sms_nuevo.withColumn("anio", year(col("fecha_envio")))

df_sms_nuevo = df_sms_nuevo.withColumn("mes", month(col("fecha_envio")))

df_sms_nuevo = df_sms_nuevo.withColumn(
    "periodo",
    date_format(col("fecha_envio"), "yyyy-MM")
)

df_sms_nuevo = df_sms_nuevo.withColumn("plataforma",lit("sms"))

df_sms_nuevo = df_sms_nuevo.withColumn("origen",lit("sms_nuevo"))

df_sms_nuevo = df_sms_nuevo.withColumnRenamed("telefono_normalizado","numero")


In [0]:
df_sms_final = df_sms_antiguo_final.union( df_sms_nuevo.select(
    col("anio"),
    col("mes"),
    col("cliente"),
    col("producto"),
    col("fecha_envio"),
    col("usuario_creacion"),
    col("numero"),
    col("control"),
    col("plataforma"),
    col("origen"),
    col("periodo"),
    col("ingestion_date")
))


In [0]:
df_sms_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.sms_transformed")